<a href="https://colab.research.google.com/github/Rajat4007/DevSync/blob/main/Gen_AI(Prompt).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -r /content/drive/MyDrive/GenAI/requirement.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.7 MB/s eta 0:00:00


#Single Message

##Static Prompt

In [ ]:
%%writefile app.py

from langchain_google_genai import ChatGoogleGenerativeAI
import streamlit as st
import os

# Read API key from environment variable
api_key = os.getenv('GEMINI_API_KEY')
print(f"DEBUG: API Key in app.py starts with: {api_key[:5] if api_key else 'None'}") # Debug print

model=ChatGoogleGenerativeAI(
  model = "gemini-3.6-flash",
  google_api_key=api_key
)

st.set_page_config(
    page_title="Research Tool",
    page_icon="🔬",
    layout="centered",
    initial_sidebar_state="auto",
    menu_items=None,
)

st.header("Research Tool")
user_input = st.text_input("Enter the Input")

if st.button("Summarize"):
    if user_input:
        st.text("Summarizing...")
        response = model.invoke(user_input)
        st.write(response.text)

    else:
        st.warning("Please enter some input to summarize.")

Overwriting app.py


In [ ]:
# Set the GEMINI_API_KEY as an environment variable for child processes
import os
os.environ['GEMINI_API_KEY'] = api_key

# Kill any previously running Streamlit processes
!pkill -f streamlit

# Launch the Streamlit app in the background, passing the API key via environment variable
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Kill any existing ngrok tunnels to ensure a clean connection
ngrok.kill()

ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_auth_token)

# Connect ngrok to the Streamlit port (8501)
public_url = ngrok.connect(8501)
print(f"Streamlit app public URL: {public_url}")

Streamlit app public URL: NgrokTunnel: "https://importer-brook-pettiness.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
print('To stop the Streamlit process, run the following command:')
# !pkill -f streamlit
# print('Streamlit process stopped.')

To stop the Streamlit process, run the following command:
Streamlit process stopped.


### Explanation of `!pkill -f streamlit`

*   **`!`**: This prefix indicates that the command following it is a shell command, not Python code, and should be executed directly in the Colab terminal.
*   **`pkill`**: This command is used to terminate processes based on their name or other attributes. It's a more powerful version of `kill` because it can kill multiple processes at once.
*   **`-f`**: This option tells `pkill` to look for the full command line of a process, not just the process name. This is useful because `streamlit` might be part of a longer command like `python -m streamlit run app.py`.
*   **`streamlit`**: This is the pattern `pkill` will search for in the command lines of running processes. Any process whose command line contains "streamlit" will be terminated.

### Using Streamlit in Google Colab: Essential Steps

When developing and running Streamlit applications in Google Colab, follow these key steps to ensure proper functionality and external access:

1.  **Save your Streamlit code as a `.py` file:** Streamlit apps are designed to run from a Python script. Use `%%writefile app.py` (or similar) to save your application code to a `.py` file in your Colab environment.

2.  **Handle API Keys via Environment Variables:**
    *   **Inside `app.py`:** Access your API key using `os.getenv('YOUR_API_KEY_NAME')` instead of `google.colab.userdata.get()`. The latter relies on the Colab kernel and won't work in a separate Streamlit process.
    *   **In Colab cell (before launching Streamlit):** Set the API key as an environment variable using `os.environ['YOUR_API_KEY_NAME'] = userdata.get('YOUR_API_KEY_NAME')` (or directly if you have the key). Then, when launching Streamlit, ensure this environment variable is passed, for example: `!YOUR_API_KEY_NAME="$YOUR_API_KEY_VALUE" streamlit run app.py`.

3.  **Launch Streamlit in the Background:** Use `!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &` to run your Streamlit app in a background process on a specific port (e.g., 8501).

4.  **Expose Streamlit to the Web using `ngrok` or `localtunnel`:**
    *   Since Streamlit runs on a local port within Colab, you need a tunneling service to access it from your web browser.
    *   **`ngrok` (recommended):** Install `pyngrok` (`!pip install pyngrok`), get an `NGROK_AUTH_TOKEN` from [ngrok.com](https://ngrok.com/signup) and save it in Colab secrets. Then use `ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))` and `public_url = ngrok.connect(8501)` to get your public URL.
    *   **`localtunnel` (alternative):** Install `localtunnel` (`!npm install -g localtunnel`) and then run `!lt --port 8501` to get a public URL.

5.  **Access the Public URL:** Once `ngrok` or `localtunnel` provides a public URL, open this URL in your web browser to interact with your Streamlit application.

### Streamlit Background Run Command Explained

```bash
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &
```

Yeh command aamtaur par **Google Colab** ya kisi Linux environment me Streamlit application ko **background me run karne**, **terminal ko free rakhne**, aur **logs ko track karne** ke liye use ki jaati hai.

---

## 📌 Detailed Breakdown (Line-by-Line)

| Component | Meaning / Purpose |
| :--- | :--- |
| `!` | **Colab Shell Escape:** Notebook cell ko batata hai ki yeh Python code nahi, balki ek **Bash / Linux shell command** hai. |
| `streamlit run app.py` | **Execution Command:** `app.py` script ko Streamlit web server ke roop me start karta hai. |
| `--server.port 8501` | **Port Configuration:** App ko explicitly port `8501` par bind karta hai (tunneling tools jaise localtunnel ya ngrok ke sath connect karne ke liye zaroori). |
| `> /content/streamlit.log` | **Standard Output Redirection (`stdout`):** App ke saare normal terminal outputs ko screen par print karne ke bajaye `/content/streamlit.log` file me write karta hai. |
| `2>&1` | **Standard Error Redirection (`stderr`):** File Descriptor 2 (Errors) ko File Descriptor 1 (Output) par redirect karta hai, jisse saare errors bhi isi log file me save ho jayein. |
| `&` | **Background Execution:** Command ko background process ke taur par run karta hai taaki Colab cell block na ho. |

---

## 🧠 Kyun Likhte Hain Aise? (Key Reasons)

1. **Cell Block Hone Se Bachana:** Agar `&` nahi lagayenge, toh Streamlit server foreground me chalega aur Colab ka cell hamesha "Running" status me phasa rahega.
2. **Next Steps Execute Karna:** Background me run hone se aap next cell me aasaani se `localtunnel` ya `ngrok` setup kar sakte hain.
3. **Debugging Aur Error Tracking:** App crash hone ya error aane par aap aasani se `!cat /content/streamlit.log` run karke logs dekh sakte hain.

##Dynamic Prompt

In [ ]:
%%writefile app.py

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import streamlit as st
import os

# Read API key from environment variable
api_key = os.getenv('GEMINI_API_KEY')


model=ChatGoogleGenerativeAI(
  model = "gemini-3.6-flash",
  google_api_key=api_key
)

st.set_page_config(
    page_title="Research Tool",
    page_icon="🔬",
    layout="centered",
    initial_sidebar_state="auto",
    menu_items=None,
)

st.header("Research Tool")

paper_input = st.selectbox( "Select Research Paper Name", ["Attention Is All You Need", "BERT: Pre-training of Deep Bidirectional Transformers", "GPT-3: Language Models are Few-Shot Learners", "Diffusion Models Beat GANs on Image Synthesis"] )

style_input = st.selectbox( "Select Explanation Style", ["Beginner-Friendly", "Technical", "Code-Oriented", "Mathematical"] )

length_input = st.selectbox( "Select Explanation Length", ["Short (1-2 paragraphs)", "Medium (3-5 paragraphs)", "Long (detailed explanation)"] )

template = PromptTemplate(
    template="""
Please summarize the research paper titled "{paper_input}" with the following specifications:
Explanation Style: {style_input}
Explanation Length: {length_input}
1. Mathematical Details:
   - Include relevant mathematical equations if present in the paper.
   - Explain the mathematical concepts using simple, intuitive code snippets where applicable.
2. Analogies:
   - Use relatable analogies to simplify complex ideas.
If certain information is not available in the paper, respond with: "Insufficient information available" instead of guessing.
Ensure the summary is clear, accurate, and aligned with the provided style and length.
""",
input_variables=['paper_input', 'style_input','length_input'],
validate_template=True
)

# Construct the final prompt using the template and selected inputs
# The result of template.invoke should be stored and used.
generated_prompt = template.invoke({
    'paper_input':paper_input,
    'style_input':style_input,
    'length_input':length_input
})

if st.button("Summarize"):
    # Since st.selectbox always returns a value, an inner 'if' condition
    # to check if input is provided is generally not needed here.
    st.text("Summarizing...")
    response = model.invoke(generated_prompt) # Use the correctly generated prompt
    st.write(response.text)


Overwriting app.py


We can make a separate file for Template use.It is shown below

In [ ]:
from langchain_core.prompts import PromptTemplate
import json # Import the standard json library

# template
template = PromptTemplate(
    template="""
Please summarize the research paper titled "{paper_input}" with the following specifications:
Explanation Style: {style_input}
Explanation Length: {length_input}
1. Mathematical Details:
   - Include relevant mathematical equations if present in the paper.
   - Explain the mathematical concepts using simple, intuitive code snippets where applicable.
2. Analogies:
   - Use relatable analogies to simplify complex ideas.
If certain information is not available in the paper, respond with: "Insufficient information available" instead of guessing.
Ensure the summary is clear, accurate, and aligned with the provided style and length.
""",
input_variables=['paper_input', 'style_input','length_input'],
validate_template=True
)

# Create a dictionary with the essential components of the prompt
# prompt_definition = {
#     "template": template.template,
#     "input_variables": template.input_variables
# }

# # Serialize this dictionary to a JSON file
# with open('template.json', 'w') as f:
#     json.dump(prompt_definition, f, indent=2) # Use json.dump for a standard JSON structure
template.save('template.json')

In [ ]:
%%writefile app.py

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate,load_prompt
import streamlit as st
import os

# Read API key from environment variable
api_key = os.getenv('GEMINI_API_KEY')


model=ChatGoogleGenerativeAI(
  model = "gemini-3.6-flash",
  google_api_key=api_key
)

st.set_page_config(
    page_title="Research Tool",
    page_icon="🔬",
    layout="centered",
    initial_sidebar_state="auto",
    menu_items=None,
)

st.header("Research Tool")

paper_input = st.selectbox( "Select Research Paper Name", ["Attention Is All You Need", "BERT: Pre-training of Deep Bidirectional Transformers", "GPT-3: Language Models are Few-Shot Learners", "Diffusion Models Beat GANs on Image Synthesis"] )

style_input = st.selectbox( "Select Explanation Style", ["Beginner-Friendly", "Technical", "Code-Oriented", "Mathematical"] )

length_input = st.selectbox( "Select Explanation Length", ["Short (1-2 paragraphs)", "Medium (3-5 paragraphs)", "Long (detailed explanation)"] )

template = load_prompt('template.json')

# Construct the final prompt using the template and selected inputs
# The result of template.invoke should be stored and used.
# generated_prompt = template.invoke({
#     'paper_input':paper_input,
#     'style_input':style_input,
#     'length_input':length_input
# })

# if st.button("Summarize"):
#     st.text("Summarizing...")
#     response = model.invoke(generated_prompt)
#     st.write(response.text)
if st.button('Summarize'): #Doo baar invoke krne se bdya hai chain bna do
    chain = template | model
    result = chain.invoke({
        'paper_input':paper_input,
        'style_input':style_input,
        'length_input':length_input
    })
    st.write(result.text)


Overwriting app.py


In [ ]:
import os
os.environ['GEMINI_API_KEY'] = api_key
!pkill -f streamlit
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Kill any existing ngrok tunnels to ensure a clean connection
ngrok.kill()

ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_auth_token)

# Connect ngrok to the Streamlit port (8501)
public_url = ngrok.connect(8501)
print(f"Streamlit app public URL: {public_url}")

Streamlit app public URL: NgrokTunnel: "https://importer-brook-pettiness.ngrok-free.dev" -> "http://localhost:8501"


#List of Messages

##Making a Normal ChatBot(`Static Prompt`)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

model=ChatGoogleGenerativeAI(
  model = "gemini-3.6-flash",
  google_api_key=api_key
)

while True:
  user_input = input("You : ");
  if(user_input == "exit"):
    break
  response = model.invoke(user_input)
  print("Bot : ",response.text)

You : hi
Bot :  Hello! How can I help you today?
You : compare 2 and 3 which is large
Bot :  **3** is larger than 2. 

When counting, 3 comes after 2, which means it represents a greater quantity ($3 > 2$).
You : now multiply larger number by 10
Bot :  Could you please share the numbers you are referring to? Once you provide them, I'll find the larger one and multiply it by 10 for you!
You : exit


Now we will use chat history so that the above problem may not arise

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')

model=ChatGoogleGenerativeAI(
  model = "gemini-3.5-flash",
  google_api_key=api_key
)
chat_history = [
    SystemMessage(content="You are a helpful assistant."),
]
while True:
  user_input = input("You : ")
  chat_history.append(HumanMessage(content=user_input))

  if(user_input.lower() == "exit"):
    break
  response = model.invoke(chat_history)
  chat_history.append(AIMessage(content=response.text))
  print("Bot : ",response.text)
print(chat_history)

You : HI
Bot :  Hello! How can I help you today?
You : compare greater between 2 and 5
Bot :  Between 2 and 5, **5** is the greater number (5 > 2).
You : now multiply greater by 10
Bot :  Multiplying the greater number (5) by 10 gives **50** (5 × 10 = 50).
You : multiply smallest by 20
Bot :  Multiplying the smallest number (2) by 20 gives **40** (2 × 20 = 40).
You : now compare the greatest number between the result 
Bot :  Comparing the two results:

*   The first result is **50**.
*   The second result is **40**.

Between 50 and 40, the greatest number is **50** (50 > 40).
You : EXIT
[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='HI', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='compare greater between 2 and 5', additional_kwargs={}, response_metadata={

### Langchain `chat_history` ke Components ka Vivaran (Explanation)

`chat_history` ek list hoti hai jismein conversation ke saare messages store kiye jaate hain. Har message ek specific type ka hota hai jo uske role ko define karta hai:

1.  **`SystemMessage`**: (System Message)
    *   **Uddeshya (Purpose)**: Yeh bot ke liye **initial instructions ya persona** set karta hai. User ko yeh message directly dikhai nahi deta, lekin yeh bot ke responses ko guide karta hai ki use kaisa behave karna hai (e.g., 'helpful assistant', 'friendly chatbot').
    *   **Example (aapke notebook se)**: `SystemMessage(content='You are a helpful assistant.')`
    *   **Revision Notes**: `SystemMessage` bot ko batata hai ki use kaisa behave karna hai. Ek chat session mein yeh generally ek hi baar shuru mein define hota hai.

2.  **`HumanMessage`**: (User Message)
    *   **Uddeshya (Purpose)**: Yeh **user ke inputs** ko represent karta hai. Jab aap `input()` function se kuch type karte hain, woh `HumanMessage` ke roop mein `chat_history` mein add ho jata hai.
    *   **Example (aapke notebook se)**: `HumanMessage(content='HI')`, `HumanMessage(content='compare greater between 2 and 5')`
    *   **Revision Notes**: `HumanMessage` woh hota hai jo user bolta hai. Har baar user input deta hai, ek naya `HumanMessage` history mein add ho jata hai.

3.  **`AIMessage`**: (AI/Bot Message)
    *   **Uddeshya (Purpose)**: Yeh **AI (bot) ke responses** ko represent karta hai. Jab `model.invoke(chat_history)` call hota hai aur bot reply karta hai, woh reply `AIMessage` ke roop mein `chat_history` mein add ho jata hai.
    *   **Example (aapke notebook se)**: `AIMessage(content='Hello! How can I help you today?')`, `AIMessage(content='Between 2 and 5, **5** is the greater number (5 > 2).')`
    *   **Revision Notes**: `AIMessage` woh hota hai jo bot reply karta hai. Har baar bot reply karta hai, ek naya `AIMessage` history mein add ho jata hai.

### `chat_history` Kaise Kaam Karta Hai?

`chat_history` ek sequence of messages maintain karta hai. Jab aap `model.invoke(chat_history)` call karte hain, toh Langchain model ko **poori conversation history** bhej deta hai. Isse model pichle messages ko **samajh pata hai aur uske context mein relevant jawab de pata hai. Is tarah, conversation ki continuity bani rehti hai.**

##Dynamic Message

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

chat_template = ChatPromptTemplate([
    SystemMessage(content="You are a helpful{domain} assistant."),
    HumanMessage(content="Explain in simple term What is {topic}.")
])

result = chat_template.invoke({'domain':'friendly','topic':'Relationship'})
print(result)

messages=[SystemMessage(content='You are a helpful{domain} assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain in simple term What is {topic}.', additional_kwargs={}, response_metadata={})]


Upr wala jo syntax hai usme `SystemMessage` aur `HumanMessage` ka syntax alg hoga. Ye jo likhe hai wo syntax `static message` ke liye hai tha.`dynamic message ` mai hme `tuple` pass krne hote hai  

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

chat_template = ChatPromptTemplate([
    ('system','You are a helpful {domain} assistant'),
    ('human','Explain in simple term what is {topic}')
])

result = chat_template.invoke({'domain':'friendly','topic':'Relationship'})
print(result)

messages=[SystemMessage(content='You are a helpful friendly assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain in simple term what is Relationship', additional_kwargs={}, response_metadata={})]


Upr dekho ki `system message` aur `human message` ka syntax bhi change hai. hamlog `ChatPromptTemplate.from_messages` bhi likh skte hai qki kahi kahi ye bhi syntax use hota hai

#Message PlaceHolder

In [ ]:

# Define the messages
human_msg = 'HumanMessage(content="I want to request a refund for my order #12345.")'
ai_msg = 'AIMessage(content="Your refund request for order #12345 has been initiated. It will be processed in 3-5 business days.")'

file_name = "chat_history.txt"

# Open the file in write mode ('w') and write the content
with open(file_name, 'w') as f:
    f.write(f"{human_msg}\n")
    f.write(f'{ai_msg}')

print(f"Content successfully written to {file_name}")

Content successfully written to chat_history.txt


In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
#chat template
chat_template = ChatPromptTemplate.from_messages([
    ('system','You are a helpful assistant'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human','{query}')
])

chat_history= []
#load chat history
with open('chat_history.txt') as f:
    chat_history.extend(f.readlines())

print(chat_history)
#create prompt
prompt = chat_template.invoke({'chat_history':chat_history,'query':'Where is my refund'})
print(prompt)

['HumanMessage(content="I want to request a refund for my order #12345.")\n', 'AIMessage(content="Your refund request for order #12345 has been initiated. It will be processed in 3-5 business days.")']
messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='HumanMessage(content="I want to request a refund for my order #12345.")\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='AIMessage(content="Your refund request for order #12345 has been initiated. It will be processed in 3-5 business days.")', additional_kwargs={}, response_metadata={}), HumanMessage(content='Where is my refund', additional_kwargs={}, response_metadata={})]
